# Notebook 5: Train/Test Split, Pipelines, PCA & Models

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split,cross_val_score,StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import *

df=pd.read_excel('customer_data_raw_advanced.xlsx').drop_duplicates()

## Stage 21

In [ ]:
X=df.drop(columns=['Target','LeakageFeature','CustomerID','FakeID'],errors='ignore')
y=df['Target']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

## Stage 22

In [ ]:
num=X_train.select_dtypes(include='number').columns
cat=X_train.select_dtypes(exclude='number').columns
prep=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]),num),('cat',Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore'))]),cat)])
pipe=Pipeline([('prep',prep),('model',LogisticRegression(max_iter=1000))])
pipe.fit(X_train,y_train)

## Stage 23

In [ ]:
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
cross_val_score(pipe,X,y,cv=cv)

## Stage 24

In [ ]:
Xn=df.select_dtypes(include='number').drop(columns=['Target','LeakageFeature'],errors='ignore').fillna(0)
Xs=StandardScaler().fit_transform(Xn)
pca=PCA(n_components=0.95)
Xp=pca.fit_transform(Xs)
Xp.shape

## Stage 25

In [ ]:
models={'LR':LogisticRegression(max_iter=1000),'DT':DecisionTreeClassifier(random_state=42),'RF':RandomForestClassifier(random_state=42)}
for n,m in models.items():
 p=Pipeline([('prep',prep),('model',m)])
 p.fit(X_train,y_train)
 pr=p.predict(X_test)
 print(n,accuracy_score(y_test,pr),f1_score(y_test,pr))

## Practice
- Compare models
- Change test size
- Try PCA with fixed components